<a href="https://colab.research.google.com/github/RobJavVar/DataSciencePsychNeuro/blob/master/ExerciseSubmissions/11_the-beauty-of-knn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 11: The beauty of kNN

In this exercise, you'll gain practice working with kNN. We'll use the [diamonds](https://ggplot2.tidyverse.org/reference/diamonds.html) dataset, which comes as part of `ggplot2`. This dataset provides information on the quality and price of 50,000 diamonds


## 1. Data, Plotting, and Train/Test Sets (2 pts)
-----
* Load the the `class` and `tidyverse` packages.
* Assign the `diamonds` data set to a simpler name. Then, create a new variable `price_bin` that splits the `price` variable into a binary variable, where 1 indicates that the diamond costs greater than the mean price, and 0 indicates that the diamond costs less than the mean price. Set `price_bin` to be a factor. (*Hint: use the if_else() function*)
* Select just the `carat`, `depth`, `table`, `x`, `y`, and your new `price_bin` variables
* Print the first few lines of the data set
* Print the dimensions of the data set


In [ ]:
library(class)
library(tidyverse)
dat <- diamonds
dat <- dat %>%
    mutate(
        price_bin = ifelse(price > mean(price, na.rm = T), 1, 0))
dat <- dat %>%
    select(carat, depth, table, x, y, price_bin)
print(head(dat))
print(dim(dat))


### Plot
Create a scatterplot of the link between `carat` and `depth`, and use the `color` aesthetics mapping to differentiate between diamonds that cost above versus below the mean price.

In [ ]:
ggplot(data = dat, aes(x = carat, y = depth, color = price_bin)) + geom_point()


Based on the above scatterplot, how do you think kNN will perform using only these two variables to predict diabetes diagnosis? Which variable, carat or depth, gives us the most information about which price class the diamond will belong to?
> * I think it will be able to predict the price class pretty accurately because all of the data are grouped up, there are a lot of nearby neighbors mostly of the same class.
>
> *


### Test vs Train

Before we run KNN on these data, we need to set aside a portion of the observations as our test set. Below, randomly divide the data such that 30% are allotted to the `test` set and the rest are allotted to the `train` set. Print the first few lines of each set, and print the dimensions of each set to double check your division of the data.

In [ ]:
set.seed(2023)

data_30 <- dat %>% sample_frac(0.3)

data_70 <- anti_join(dat, data_30)

dim(data_30)
dim(data_70)
dim(dat)


## 2: KNN (3 points)
----
Now, use the `knn()` function from the `class` library to predict `price_bin` from the `carat` and `depth`. Set `k = 3`.

*Hint: Review the format required for the arguments of knn()*

In [ ]:
set.seed(2023)
train_x <- data_70[, c("carat", "depth")]
test_x  <- data_30[, c("carat", "depth")]
train_y <- data_70$price_bin
knn_model <- knn(train = train_x, 
                 test = test_x, 
                 cl = train_y, 
                 k = 3)


Now, output a confusion matrix and calculate the test error to evaluate model performance.

In [ ]:
confusion_df <- data.frame(predicted = knn_model,actual = data_30$price_bin)
table(confusion_df)
print(paste("Accuracy:",mean(confusion_df$predicted == confusion_df$actual)))


How did your model perform?
> * Very well, 95% accuracy. It seems to be an even proportion of false positive and false negatives relative to the count of actual positives and negatives.
>
> * Notably, there were far more actual points below the mean price, indicating a positive skew of the data.


Let's try to improve our model by adding all of the other variables in our data set as predictors. Rerun your `knn()` below, keeping `k = 3`. Again, output a confusion matrix and error rate for your updated model fit.

In [ ]:
set.seed(2023)
knn_model2 <- knn(train = data_70, 
                 test = data_30, 
                 cl = train_y, 
                 k = 3)
confusion_df <- data.frame(predicted = knn_model2,actual = data_30$price_bin)
table(confusion_df)
print(paste("Accuracy:",mean(confusion_df$predicted == confusion_df$actual)))


Did your model predictions improve?
> * Yes, it was far more accurate, with an accuracy of 99.94%.
>
> *


# 3: for loop (3 points)
----

So adding additional predictors didn't shift our error much. Let's see if adjusting `k` has a larger impact on model accuracy.

Using your initial model above with just `carat` and `depth`, run a `for loop` that runs the same model 30 times, for `k = 1:30`.

Output a data frame that has `k` and the overall `error` as columns.

The structure of the output data frame and `for loop` are provided for you below. Note that your loop will take a minute or two to run because there are so many observations in the dataset. It may be helpful while you are writing and testing your loop to run it on a subset of the data with only a handful of rows.

In [ ]:
# this is provided
# setting up empty table to store for loop output
output  <- data.frame(k = seq(1:30),
                     error = rep(NA, 30))
head(output)


In [ ]:
for (k in seq(1:30)) {
    knn_fits  <- knn(train = train_x, 
                 test = test_x, 
                 cl = train_y, 
                 k = k)

    #overall error
    confusion_df <- data.frame(predicted = knn_fits,actual = data_30$price_bin)
    accuracy <- mean(confusion_df$predicted == confusion_df$actual)
    output$error[k]  <- (1 - accuracy)

}
head(output)
table(output)


Create a line plot of your `output` object using `ggplot`. Add a (non-linear) `geom_smooth` layer.

In [ ]:
ggplot(data = output, aes(x = k, y = error)) + geom_point() + geom_smooth() + ylim(0, 0.1)


Interpret your plot. What would you select as the best value of `k`? How much does this improve your test error?
> * I would select k = 9, but it only improved the test error by about 0.3%
>
> *


# 4: Standardizing predictors (2)
-----

Because knn is based on distances between points, it is very sensitive to the scale of your variables. Looking at our predictor variables, we can see that `carat` and `depth` are orders of magnitude different in terms of scales. Maybe we can improve our fit even more by addressing this!

Below, use the `scale()` function to standardize your predictors. (Note that you don't need to standardize `price_bin`.)

Then, run your model a final time with your standardized predictors (just `carat` and `depth` still). Set `k` to the optimal value you determined in your plot above. Output the confusion matrix and error rate again.

In [ ]:
set.seed(2023)
train_x_scaled <- scale(train_x)
train_attr <- attributes(train_x_scaled)
test_x_scaled <- scale(test_x, 
                       center = train_attr$`scaled:center`, 
                       scale = train_attr$`scaled:scale`)
knn_final_model <- knn(train = train_x_scaled, 
                 test = test_x_scaled, 
                 cl = as.factor(train_y), 
                 k = 9)

confusion_df_final <- data.frame(predicted = knn_final_model,actual = data_30$price_bin)
table(confusion_df_final)
print(paste("Accuracy:",mean(confusion_df_final$predicted == confusion_df_final$actual)))


What impact did rescaling the data have on your error rate?
> * It is the same as before... maybe I did something wrong
>
> *


**DUE:** 5pm March 17, 2026

**IMPORTANT** Did you collaborate with anyone on this assignment? If so, list their names here.
> I was worried when my error didn't change so I asked Gemini if I did something wrong in the last code block but it said everything looked fine.
>
>
